In [ ]:
from sklearn.neighbors import KDTree
import pandas as pd

In [ ]:
fp = "../data/sba_loans_prepared/sba_loans_num_enc_train.csv"
df_train = pd.read_csv(fp)

In [ ]:
preds = [ c for c in df_train.columns.tolist() if c != "LoanStatus"]
X_train_df = df_train[preds]

In [ ]:
from sklearn.neighbors import KDTree

kdt = KDTree(X_train_df, leaf_size=30, metric='euclidean')

In [ ]:
bad_borr_sel = (df_train.LoanStatus == 1)

In [ ]:
bad_borr_sel

In [ ]:
NUM_NBRS = 3
X_bad_borr_df = df_train[bad_borr_sel][preds]

In [ ]:
bdist, bind = kdt.query(X_bad_borr_df, k=NUM_NBRS)

In [ ]:
bind = bind.flatten().tolist() + df_train[bad_borr_sel].index.tolist()

In [ ]:
df_bad_borr_nbrh = df_train[df_train.index.isin(bind)]
bad_borr_ind = df_bad_borr_nbrh.index.tolist()
df_bad_borr_info = pd.DataFrame(bad_borr_ind)
df_bad_borr_info.columns = ["bad_borrower_index"]
fpbb = "../data/sba_loans_prepared/bad_borr_index.csv"
df_bad_borr_info.to_csv(fpbb, index=False)

In [ ]:
df_bad_borr_nbrh.LoanStatus.value_counts()

In [ ]:
from sklearn.neighbors import NeighborhoodComponentsAnalysis

In [ ]:
nca = NeighborhoodComponentsAnalysis(random_state=42)

In [ ]:
X_df = df_bad_borr_nbrh[preds]
Y = df_bad_borr_nbrh["LoanStatus"]

In [ ]:
X_df.shape

In [ ]:
# this can take some time - a few minutes.
nca.fit(X_df, Y)

In [ ]:
X_trans_train = nca.transform(X_df)
df_train_nca_enc = pd.DataFrame(X_trans_train)
df_train_nca_enc.columns = ["nca-" + str(i+1) for i in range(df_train_nca_enc.shape[1])]
df_train_nca_enc["LoanStatus"] = Y.values

In [ ]:
fp =  "../data/sba_loans_prepared/sba_loans_nca_enc_train.csv"
df_train_nca_enc.to_csv(fp, index=False)

In [ ]:
fp =  "../data/sba_loans_prepared/sba_loans_num_enc_val.csv"
df_val = pd.read_csv(fp)

In [ ]:
df_val

In [ ]:
X_val = df_val[preds]
X_trans_val = nca.transform(X_val)

In [ ]:
df_val_nca_enc = pd.DataFrame(X_trans_val)
nca_preds = ["nca-" + str(i+1) for i in range(df_val_nca_enc.shape[1])]
df_val_nca_enc.columns = nca_preds
df_val_nca_enc["LoanStatus"] = df_val["LoanStatus"].values
fp =  "../data/sba_loans_prepared/sba_loans_nca_enc_val.csv"
df_val_nca_enc.to_csv(fp, index=False)

In [ ]:
df_val["LoanStatus"].values

In [ ]:
fp =  "../data/sba_loans_prepared/sba_loans_num_enc_test.csv"
df_test = pd.read_csv(fp)

In [ ]:
X_test = df_test[preds]
X_trans_test = nca.transform(X_test)
df_test_nca_enc = pd.DataFrame(X_trans_test)
df_test_nca_enc.columns = nca_preds
df_test_nca_enc["LoanStatus"] = df_test["LoanStatus"].values
fp =  "../data/sba_loans_prepared/sba_loans_nca_enc_test.csv"
df_test_nca_enc.to_csv(fp, index=False)

In [ ]:
df_test_nca_enc.LoanStatus.value_counts()